In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os
import pickle
import random

In [3]:
with open('data.txt', 'r', encoding='utf-8') as f:
    text_data = f.read()

In [4]:
text_data

"my name is sudhanshu kumar , i work with euron , Helping Millions of Students Succeed\nSudhanshu's commitment to affordable education wasn't just a business strategy—it was his life's mission. Over the years, iNeuron has helped over 1.5 million students from 34+ countries, providing them with the skills they need to succeed in today's competitive job market. Many of these students, like Sudhanshu himself, came from disadvantaged backgrounds. They saw iNeuron as a lifeline—an opportunity to rise above their circumstances.\n\nIn 2022, iNeuron was acquired by PhysicsWallah in a deal worth ₹250 crore. While this acquisition was a significant milestone, Sudhanshu remained focused on his mission. Even after the acquisition, iNeuron continued to offer some of the most affordable and accessible tech courses in the world.\nHelping Millions of Students Succeed\nSudhanshu's commitment to affordable education wasn't just a business strategy—it was his life's mission. Over the years, iNeuron has h

In [5]:
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=50000, oov_token='<OOV>')

tokenizer.fit_on_texts([text_data])

In [6]:
tokenizer

In [10]:
sequence = tokenizer.texts_to_sequences([text_data])[0]

In [11]:
len(sequence)

860

In [12]:
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [22]:
max_seq_length = 100
def create_dataset(seq,window_size = max_seq_length):
    input , lable = [],[]
    for i in range(len(seq)- window_size):
        input.append(seq[i:i+window_size])
        lable.append(seq[i+1:i+window_size+1])
    return np.array(input) , np.array(lable)


In [23]:
x_data,y_data = create_dataset(sequence)

In [24]:
x_data[0]

array([158, 159,  16,   9, 160, 161, 162,  20,  21,  69,  49,   5,  22,
        28,  36,  70,   2,  23,  10,  71,  29,   4,  72,  73,  12,   8,
        74,  24,  30,   3,  50,  11,  75,  76,  30,  77,  78,  51,  22,
        14,  79,  80,  81,  82,  20,   3,  31,  37,  83,   2,  28,   6,
        84,  85,  86,  87,  38,   5,  88,  22,  39,   9,  52,  53,  14,
        54,  40,  37,  89,  11,  90,   4,  91,  25,   2,  92,  93,  18,
        94,   6,  95,  11,  12,  96,  26,  97,   6,   4,  98,  99, 100,
       101,  55,  27,  41,  12,   4, 102, 103,   9])

In [25]:
len(x_data[0])

100

In [26]:
y_data[0]

array([159,  16,   9, 160, 161, 162,  20,  21,  69,  49,   5,  22,  28,
        36,  70,   2,  23,  10,  71,  29,   4,  72,  73,  12,   8,  74,
        24,  30,   3,  50,  11,  75,  76,  30,  77,  78,  51,  22,  14,
        79,  80,  81,  82,  20,   3,  31,  37,  83,   2,  28,   6,  84,
        85,  86,  87,  38,   5,  88,  22,  39,   9,  52,  53,  14,  54,
        40,  37,  89,  11,  90,   4,  91,  25,   2,  92,  93,  18,  94,
         6,  95,  11,  12,  96,  26,  97,   6,   4,  98,  99, 100, 101,
        55,  27,  41,  12,   4, 102, 103,   9, 104])

In [27]:
len(y_data[0])

100

In [28]:
class PositionalEncoding(layers.Layer):
    def __init__(self, max_len, d_model):
        super().__init__()
        pos = np.arange(max_len)[:, np.newaxis]
        i = np.arange(d_model)[np.newaxis, :]
        angle_rates = 1 / np.power(10000, (2 * (i // 2)) / np.float32(d_model))
        angle_rads = pos.angle_rates      # Broadcasting to get the angles for each position and dimension
        angle_rads[:, 0::2] = np.sin(angle_rads[:, 0::2])   # For even index in the array, apply sin to the angle
        angle_rads[:, 1::2] = np.cos(angle_rads[:, 1::2])   # For odd index in the array, apply cos to the angle
        self.pos_encoding = tf.cast(angle_rads[np.newaxis, ...], tf.float32)
        
    def call(self, x):
        return x + self.pos_encoding[:, :tf.shape(x)[1], :]

In [ ]:
def transformer_block(embed_dim , num_heads , ff_dim , dropout = 0.1):
    layers.Input